# Lecture 20: Pandas: Grouping, Merging, and Reshaping

## Learning Objectives

- Group data with groupby() and aggregate with multiple functions
- Merge DataFrames using inner, left, right, and outer joins
- Concatenate DataFrames with concat()
- Reshape data with pivot_table() and melt()
- Use stack() and unstack() for index/column pivoting
- Work with multi-level indexes

## Key Topics

- groupby() + .agg() with multiple functions
- merge(): inner, left, right, outer joins
- concat() for stacking
- pivot_table() and melt()
- stack() and unstack()
- Working with multi-level indexes

### GroupBy: Split-Apply-Combine

The `groupby()` method implements the split-apply-combine pattern. You split the data into groups based on one or more columns, apply a function to each group independently, and combine the results into a new DataFrame. This is the foundation of all grouped operations in Pandas.

The `.agg()` method extends this by allowing multiple aggregation functions at once. You can pass a dictionary mapping column names to functions, or a list of functions to apply to each column. Common aggregations include `'sum'`, `'mean'`, `'count'`, `'min'`, `'max'`, and `'std'`.

In [ ]:
import pandas as pd
import numpy as np

# Sales data
sales = pd.DataFrame({
    'Region': ['North', 'South', 'North', 'South', 'North', 'South'],
    'Product': ['A', 'A', 'B', 'B', 'A', 'B'],
    'Revenue': [100, 200, 150, 250, 120, 180],
    'Quantity': [10, 15, 12, 20, 11, 14]
})
print('Sales data:')
print(sales)

In [ ]:
# groupby with multiple aggregations
grouped = sales.groupby('Region').agg({
    'Revenue': ['sum', 'mean'],
    'Quantity': ['sum', 'count']
})
print('Grouped by Region:')
print(grouped)

# Group by multiple columns
grouped2 = sales.groupby(['Region', 'Product']).agg('sum')
print('\nGrouped by Region and Product:')
print(grouped2)

In [ ]:
# Using named aggregation
result = sales.groupby('Region').agg(
    total_revenue=('Revenue', 'sum'),
    avg_quantity=('Quantity', 'mean'),
    num_orders=('Revenue', 'count')
).reset_index()
print('Named aggregation:')
print(result)

### Merging DataFrames: Joins

Real-world data is rarely in a single table. Merging (or joining) combines DataFrames based on a common key column. `pd.merge()` supports four types of joins. An inner join keeps only rows with matching keys in both tables. A left join keeps all rows from the left table, filling NaN where the right table has no match. Right and outer joins behave analogously.

Merging is fundamental to relational data analysis. For example, you might have a `customers` table and an `orders` table, and you need to merge them on `customer_id` to analyse customer behaviour. Choosing the wrong join type can lose or duplicate data, so understanding the semantics of each join is essential.

In [ ]:
# Merging DataFrames
customers = pd.DataFrame({
    'CustomerID': [1, 2, 3, 4],
    'Name': ['Alice', 'Bob', 'Charlie', 'Diana'],
    'City': ['NYC', 'London', 'Paris', 'Tokyo']
})

orders = pd.DataFrame({
    'OrderID': [101, 102, 103, 104],
    'CustomerID': [1, 2, 2, 5],
    'Amount': [250, 180, 320, 90]
})

print('Customers:')
print(customers)
print('\nOrders:')
print(orders)

In [ ]:
# Different join types
inner = pd.merge(customers, orders, on='CustomerID', how='inner')
print('Inner join:')
print(inner)

left = pd.merge(customers, orders, on='CustomerID', how='left')
print('\nLeft join:')
print(left)

outer = pd.merge(customers, orders, on='CustomerID', how='outer')
print('\nOuter join:')
print(outer)

In [ ]:
# concat for stacking
q1 = pd.DataFrame({'Product': ['A', 'B'], 'Sales': [100, 150]})
q2 = pd.DataFrame({'Product': ['A', 'B'], 'Sales': [120, 160]})

stacked = pd.concat([q1, q2], ignore_index=True)
print('Stacked vertically:')
print(stacked)

# concat with keys (creates MultiIndex)
stacked_keys = pd.concat([q1, q2], keys=['Q1', 'Q2'])
print('\nStacked with keys:')
print(stacked_keys)

### pivot_table() and melt()

`pivot_table()` creates a spreadsheet-style summary table. You specify the values to aggregate, the index (rows), columns, and aggregation function. It is essentially a multidimensional version of `groupby()`. `melt()` does the reverse: it unpivots a wide table into a long format, which is often required for plotting libraries like seaborn.

These two operations are complementary. `pivot_table()` makes data more compact and readable for human consumption, while `melt()` makes it suitable for machine learning and visualisation pipelines.

In [ ]:
# pivot_table
sales2 = pd.DataFrame({
    'Region': ['North', 'North', 'South', 'South'],
    'Product': ['A', 'B', 'A', 'B'],
    'Quarter': ['Q1', 'Q1', 'Q2', 'Q2'],
    'Revenue': [100, 150, 200, 250]
})

pivot = pd.pivot_table(sales2,
                       values='Revenue',
                       index='Region',
                       columns='Quarter',
                       aggfunc='sum')
print('Pivot table:')
print(pivot)

In [ ]:
# melt: unpivot
wide = pd.DataFrame({
    'Region': ['North', 'South'],
    'Q1': [100, 200],
    'Q2': [150, 250]
})

long = wide.melt(id_vars='Region',
                 value_vars=['Q1', 'Q2'],
                 var_name='Quarter',
                 value_name='Revenue')
print('Melted (long format):')
print(long)

### stack(), unstack(), and Multi-Level Indexes

Multi-level indexes (also called hierarchical indexes) let you work with higher-dimensional data in a 2D DataFrame. `stack()` pivots columns into row index levels, making the DataFrame longer. `unstack()` does the opposite, moving inner index levels to columns.

These operations are particularly useful when you have grouped or pivoted data and need to rearrange it for a specific analysis or visualisation. They work seamlessly with the MultiIndex that `groupby()` and `pivot_table()` produce.

In [ ]:
# stack and unstack
arrays = [['A', 'A', 'B', 'B'], ['X', 'Y', 'X', 'Y']]
index = pd.MultiIndex.from_arrays(arrays, names=['Product', 'Store'])
df_multi = pd.DataFrame({'Sales': [100, 150, 200, 130]}, index=index)
print('MultiIndex DataFrame:')
print(df_multi)

unstacked = df_multi.unstack()
print('\nUnstacked:')
print(unstacked)

stacked = unstacked.stack()
print('\nStacked back:')
print(stacked)

In [ ]:
# Working with multi-level indexes
sales3 = pd.DataFrame({
    'Region': ['North', 'North', 'South', 'South'],
    'Product': ['A', 'B', 'A', 'B'],
    'Revenue': [100, 150, 200, 130]
})

grouped = sales3.groupby(['Region', 'Product']).sum()
print('Grouped (MultiIndex):')
print(grouped)
print('\nIndex levels:', grouped.index.names)

# Access specific cross-section
print('\nNorth region only:')
print(grouped.loc['North'])

## Data Science Connection

Grouping, merging, and reshaping are the core data manipulation skills in data science. GroupBy operations power every 'aggregate by category' analysis. Merging is how you combine data from multiple sources. Reshaping with pivot tables and melt transforms data between formats suitable for analysis, visualisation, and machine learning pipelines. These skills transfer directly to SQL and Spark DataFrames.